# Phần 5 – Câu 3: Chỉ số nào liên hệ rõ nhất với biến động giá cổ phiếu?
Nhóm 5 PTDL · Trực quan hóa & Phân tích thống kê

**Dữ liệu:** bảng 4 – `roa`, `roe`, `nim`, `npl`, `car`, `ldr`, `car_nguon`, `ty_suat_sinh_loi_quy` (lợi suất cổ phiếu trong quý, %).

| Bước | Việc trong kế hoạch |
|---|---|
| 1 | Tương quan Pearson (tuyến tính) và Spearman (theo hạng, chịu được ngoại lai) giữa 6 chỉ số và lợi suất quý, 160 quan sát |
| 2 | p-value và hiệu chỉnh p-value vì kiểm định 6 chỉ số cùng lúc (Holm) |
| 3 | Riêng CAR: tính thêm trên các quý có số công bố |
| 4 | Xếp thứ tự 6 chỉ số theo độ mạnh tương quan |
| 5 | Kiểm tra lại bằng các cách khác (thêm so với kế hoạch, vì kết quả chính rất yếu) |
| 6 | Biểu đồ: heatmap, scatter kèm đường hồi quy |
| 7 | Xuất `tuong_quan.csv`, trả lời câu 3 |

**Lưu ý diễn giải:** tương quan không phải nhân quả. BCTC quý công bố *sau* khi quý kết thúc, nên đây là quan hệ **cùng kỳ** (chỉ số quý t với lợi suất quý t), không phải dự báo – phần dự báo thuộc câu 4.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
from src import config as cf
from src.data_loader import doc_tong_hop
from src.thong_ke import (tuong_quan_voi_loi_suat, loai_bien_dong_chung, thay_doi_so_voi_quy_truoc,
                          ty_trong_bien_dong_chung, BIEN_LOI_SUAT)
from src.ket_qua import xuat_ket_qua
from src.viz import (ve_ma_tran_tuong_quan, ve_cot_tuong_quan, ve_tuong_quan_so_sanh, ve_phan_tan,
                     ve_hop_nhom)

b4 = doc_tong_hop()
CS = cf.DS_CHI_SO
Y = BIEN_LOI_SUAT
COT_HIEN = ["chi_so", "so_quan_sat", "pearson_r", "pearson_p", "pearson_p_hieu_chinh",
            "spearman_rho", "spearman_p", "spearman_p_hieu_chinh", "xep_hang_do_manh"]

LUU_HINH = True
def hien(fig, ten=None, show=True):
    global LUU_HINH
    if show:
        fig.show()
    if LUU_HINH and ten:
        try:
            cf.luu_hinh(fig, ten)
        except Exception as e:
            LUU_HINH = False
            print("Không lưu được PNG (tạm tắt lưu hình):", str(e).splitlines()[0])

---
## 1–4. Phân tích chính: 6 chỉ số × lợi suất quý (160 quan sát)

**Cách đọc hệ số tương quan** (r hoặc ρ, từ −1 đến 1): dấu dương = chỉ số cao thì lợi suất cao; |hệ số| < 0,1 gần như không liên hệ, 0,1–0,3 yếu, 0,3–0,5 vừa, > 0,5 mạnh.

**Vì sao hiệu chỉnh p-value?** Kiểm định 6 chỉ số cùng lúc thì dù không chỉ số nào thật sự liên hệ, xác suất "tình cờ" có ít nhất một chỉ số p < 0,05 là 1 − 0,95⁶ ≈ 26%. Phương pháp Holm nâng p-value lên để giữ tổng sai lầm ở 5%. Một chỉ số chỉ được coi là có ý nghĩa nếu **p đã hiệu chỉnh < 0,05**.

In [2]:
tq_chinh = tuong_quan_voi_loi_suat(b4)
tq_chinh[COT_HIEN].sort_values("xep_hang_do_manh").round(3)

,chi_so,so_quan_sat,pearson_r,pearson_p,pearson_p_hieu_chinh,spearman_rho,spearman_p,spearman_p_hieu_chinh,xep_hang_do_manh
1,roe,160,0.086,0.281,1.0,0.080,0.316,1.0,1
5,ldr,160,0.037,0.644,1.0,-0.045,0.569,1.0,2
3,npl,160,0.003,0.970,1.0,0.037,0.647,1.0,3
0,roa,160,0.090,0.259,1.0,0.030,0.710,1.0,4
4,car,160,0.007,0.932,1.0,-0.022,0.783,1.0,5
2,nim,160,0.085,0.286,1.0,-0.012,0.880,1.0,6


In [3]:
hien(ve_cot_tuong_quan(tq_chinh), "p5_cot_tuong_quan_chinh")

Không lưu được PNG (tạm tắt lưu hình): 


**Kết quả:** cả 6 chỉ số đều có tương quan **rất yếu** với lợi suất cùng quý (|r|, |ρ| ≤ 0,09) và **không chỉ số nào có ý nghĩa thống kê**, kể cả trước khi hiệu chỉnh (p nhỏ nhất = 0,26). Xếp theo độ mạnh Spearman, ROE đứng đầu (ρ = 0,08); theo Pearson, ROA, ROE và NIM xấp xỉ nhau (r ≈ 0,09). Khác biệt giữa các chỉ số nhỏ đến mức thứ tự xếp hạng gần như không có ý nghĩa.

### Ma trận tương quan (heatmap)
Hàng/cột cuối là lợi suất quý. Các ô giữa 6 chỉ số với nhau cho thấy chỉ số nào "đi cùng nhau" – hữu ích cho Quân khi chọn biến cho mô hình (tránh đưa hai biến gần như trùng nhau).

In [4]:
hien(ve_ma_tran_tuong_quan(b4, phuong_phap="spearman"), "p5_ma_tran_spearman")
hien(ve_ma_tran_tuong_quan(b4, phuong_phap="pearson"), "p5_ma_tran_pearson", show=False)

Hàng "Lợi suất quý" gần như trắng toàn bộ (hệ số quanh 0). Trong khi đó một số chỉ số liên hệ khá mạnh với nhau: ROA–NIM (ρ = 0,68), ROA–CAR (0,66), NIM–CAR (0,66), ROA–ROE (0,61) – do đặc điểm mô hình kinh doanh của từng ngân hàng (ví dụ VPB, TCB cùng cao NIM, ROA và CAR; ba ngân hàng quốc doanh cùng thấp). Nếu Quân đưa cả ROA, NIM, CAR vào mô hình thì cần lưu ý chúng mang thông tin trùng lặp.

### Scatter kèm đường hồi quy
ROE (xếp đầu theo Spearman) và ROA (xếp đầu theo Pearson). Mỗi chấm = 1 ngân hàng–quý.

In [5]:
hien(ve_phan_tan(b4, "roe", ghi_chu="Đường xám = hồi quy OLS chung; gần như nằm ngang → ROE hầu như không giải thích được lợi suất"),
     "p5_phan_tan_roe")
hien(ve_phan_tan(b4, "roa"), "p5_phan_tan_roa")

Các chấm trải đều theo chiều dọc ở mọi mức ROE: cùng một mức ROE ~20%, lợi suất quý có thể từ −25% đến +50%. Chấm STB Q4/2025 (ROE −18%) nằm lẻ bên trái nhưng lợi suất quý đó vẫn dương.

---
## 5. Kiểm tra lại: vì sao tương quan yếu, và kết luận có thay đổi khi đổi cách tính?

### 5.1 Phần lớn biến động giá là biến động chung của cả ngành
Nếu trong một quý cả 8 cổ phiếu ngân hàng cùng tăng hoặc cùng giảm (do thị trường chung), thì chỉ số riêng của từng ngân hàng khó giải thích được lợi suất. Đo bằng tỷ trọng phương sai lợi suất được giải thích bởi "quý nào":

In [6]:
ty_trong = ty_trong_bien_dong_chung(b4)
print(f"{ty_trong:.0%} biến động lợi suất quý là biến động CHUNG của cả 8 ngân hàng trong cùng quý.")
print(f"Chỉ {1 - ty_trong:.0%} là phần riêng của từng ngân hàng – phần mà chỉ số tài chính riêng có thể giải thích.")
b4.groupby("ky")[Y].agg(["mean", "min", "max"]).round(1).T

66% biến động lợi suất quý là biến động CHUNG của cả 8 ngân hàng trong cùng quý.
Chỉ 34% là phần riêng của từng ngân hàng – phần mà chỉ số tài chính riêng có thể giải thích.


ky,2021Q1,2021Q2,2021Q3,2021Q4,2022Q1,2022Q2,2022Q3,2022Q4,2023Q1,2023Q2,2023Q3,2023Q4,2024Q1,2024Q2,2024Q3,2024Q4,2025Q1,2025Q2,2025Q3,2025Q4
mean,17.0,34.6,-13.5,9.0,3.8,-21.2,-5.0,1.6,13.2,5.7,3.5,-1.1,23.8,-7.4,11.7,-0.2,5.4,4.4,24.8,-3.6
min,-10.3,10.0,-25.5,-2.2,-4.3,-32.2,-11.3,-20.5,6.7,-6.2,-0.9,-8.8,2.9,-17.0,3.6,-8.6,-1.0,-10.9,8.8,-7.9
max,37.4,54.0,-2.7,22.6,17.3,-8.9,1.2,17.5,19.7,14.1,9.3,10.3,49.5,1.3,19.2,10.6,11.6,24.4,68.1,2.6


### 5.2 Bốn cách kiểm tra lại
| Cách | Ý tưởng | Câu hỏi trả lời |
|---|---|---|
| Bỏ biến động chung | Trừ trung bình 8 NH trong cùng quý cho cả chỉ số và lợi suất | Trong cùng một quý, NH có chỉ số tốt hơn bạn bè có tăng giá hơn bạn bè không? |
| Thay đổi chỉ số | Dùng mức thay đổi chỉ số so với quý trước | Chỉ số *cải thiện* thì giá có tăng không? |
| Chỉ quý 4 (2021–2024) | 32 quan sát có CAR công bố thật (theo kế hoạch) | So sánh công bằng giữa CAR và 5 chỉ số khác trên cùng mẫu |
| Cấp ngân hàng | Trung bình cả giai đoạn mỗi NH (8 quan sát) | NH có chỉ số cao có lợi suất trung bình cao hơn không? |

In [7]:
tq_bo_chung = tuong_quan_voi_loi_suat(loai_bien_dong_chung(b4, CS + [Y]),
                                      mau_du_lieu="loai_bien_dong_chung", phan_tich="kiem_tra_lai")
tq_thay_doi = tuong_quan_voi_loi_suat(thay_doi_so_voi_quy_truoc(b4, CS),
                                      mau_du_lieu="thay_doi_chi_so", phan_tich="kiem_tra_lai")
q4 = b4[(b4["quy"] == "Q4") & (b4["nam"] <= 2024)]
tq_q4 = tuong_quan_voi_loi_suat(q4, mau_du_lieu="q4_2021_2024", phan_tich="kiem_tra_lai")
tq_car = tuong_quan_voi_loi_suat(b4[b4["car_cong_bo"]], ["car"], mau_du_lieu="car_cong_bo", phan_tich="kiem_tra_lai")

nh = b4.astype({"ma_nh": str}).groupby("ma_nh")[["roa", "roe", "nim", "npl", "ldr", Y]].mean()
nh["car"] = b4[b4["car_cong_bo"]].astype({"ma_nh": str}).groupby("ma_nh")["car"].mean()
tq_nh = tuong_quan_voi_loi_suat(nh, mau_du_lieu="cap_ngan_hang", phan_tich="kiem_tra_lai")

tq_tong = pd.concat([tq_chinh, tq_bo_chung, tq_thay_doi, tq_q4, tq_car, tq_nh], ignore_index=True)
hien(ve_tuong_quan_so_sanh(tq_tong), "p5_tuong_quan_so_sanh")

In [8]:
for ten, t in [("Bỏ biến động chung", tq_bo_chung), ("Thay đổi chỉ số", tq_thay_doi),
                ("Chỉ quý 4 2021–2024", tq_q4), ("Cấp ngân hàng", tq_nh)]:
    print("■", ten)
    display(t[COT_HIEN].sort_values("xep_hang_do_manh").round(3))

■ Bỏ biến động chung


,chi_so,so_quan_sat,pearson_r,pearson_p,pearson_p_hieu_chinh,spearman_rho,spearman_p,spearman_p_hieu_chinh,xep_hang_do_manh
1,roe,160,0.114,0.151,0.604,0.121,0.128,0.768,1
0,roa,160,0.131,0.099,0.493,0.082,0.305,1.000,2
2,nim,160,0.148,0.061,0.368,0.075,0.346,1.000,3
4,car,160,0.002,0.975,1.000,-0.039,0.623,1.000,4
5,ldr,160,0.095,0.235,0.704,-0.006,0.939,1.000,5
3,npl,160,0.016,0.838,1.000,0.003,0.965,1.000,6


■ Thay đổi chỉ số


,chi_so,so_quan_sat,pearson_r,pearson_p,pearson_p_hieu_chinh,spearman_rho,spearman_p,spearman_p_hieu_chinh,xep_hang_do_manh
5,ldr,152,0.202,0.013,0.077,0.130,0.111,0.663,1
0,roa,152,0.125,0.124,0.622,0.059,0.468,1.000,2
2,nim,152,0.010,0.901,1.000,-0.049,0.546,1.000,3
3,npl,152,-0.005,0.951,1.000,0.047,0.568,1.000,4
1,roe,152,0.066,0.417,1.000,0.043,0.596,1.000,5
4,car,152,-0.064,0.435,1.000,-0.021,0.798,1.000,6


■ Chỉ quý 4 2021–2024


,chi_so,so_quan_sat,pearson_r,pearson_p,pearson_p_hieu_chinh,spearman_rho,spearman_p,spearman_p_hieu_chinh,xep_hang_do_manh
4,car,32,-0.528,0.002,0.011,-0.515,0.003,0.015,1
2,nim,32,-0.454,0.009,0.045,-0.477,0.006,0.029,2
5,ldr,32,-0.384,0.030,0.091,-0.433,0.013,0.054,3
0,roa,32,-0.412,0.019,0.077,-0.333,0.062,0.187,4
3,npl,32,-0.198,0.277,0.553,-0.106,0.563,1.000,5
1,roe,32,0.026,0.889,0.889,0.064,0.729,1.000,6


■ Cấp ngân hàng


,chi_so,so_quan_sat,pearson_r,pearson_p,pearson_p_hieu_chinh,spearman_rho,spearman_p,spearman_p_hieu_chinh,xep_hang_do_manh
3,npl,8,0.433,0.284,1.000,0.595,0.120,0.717,1
2,nim,8,0.597,0.118,0.707,0.595,0.120,0.717,1
1,roe,8,-0.228,0.588,1.000,-0.381,0.352,1.000,3
4,car,8,0.308,0.457,1.000,0.286,0.493,1.000,4
5,ldr,8,0.347,0.399,1.000,0.238,0.570,1.000,5
0,roa,8,0.284,0.495,1.000,0.214,0.610,1.000,6


**Đọc kết quả:**
- **Bỏ biến động chung:** hệ số tăng nhẹ (NIM r = 0,15; ROA 0,13; ROE ρ = 0,12) nhưng vẫn yếu và không có ý nghĩa sau hiệu chỉnh. Nhóm sinh lời (ROA, ROE, NIM) vẫn đứng đầu.
- **Thay đổi chỉ số:** LDR tăng so với quý trước đi cùng lợi suất cao hơn (r = 0,20, p = 0,013), nhưng sau hiệu chỉnh p = 0,077 > 0,05 → chưa đủ cơ sở.
- **Chỉ quý 4 (32 quan sát):** CAR (ρ = −0,52) và NIM (ρ = −0,48) có ý nghĩa ngay cả sau hiệu chỉnh, LDR (−0,43) gần ngưỡng. Đây là mẫu duy nhất có kết quả có ý nghĩa – xem 5.3.
- **Cấp ngân hàng:** NIM và NPL có ρ = 0,60 nhưng chỉ 8 quan sát, p hiệu chỉnh = 0,72 → không có ý nghĩa.

### 5.3 Tương quan âm của CAR ở quý 4: quan hệ thật hay trùng hợp?
Kế hoạch yêu cầu tính riêng CAR trên các quý có số công bố. Kết quả ρ = −0,52 (p hiệu chỉnh 0,015) nhìn thì đáng chú ý, nhưng:
1. **Không riêng CAR:** trên cùng 32 quý 4, NIM (−0,48) và LDR (−0,43) cũng âm gần bằng. Nếu CAR thật sự liên hệ với giá, nó phải nổi bật hơn các chỉ số khác.
2. **Thời điểm:** CAR năm Y công bố cùng báo cáo năm (khoảng tháng 3 năm Y+1), *sau* quý 4 – lúc giá quý 4 hình thành, thị trường chưa biết con số này.
3. **Thực chất là khác biệt nhóm:** ở 3/4 quý 4 (2021–2023), cổ phiếu quốc doanh tăng mạnh hơn tư nhân – rõ nhất Q4/2022 (+13,5% so với −5,5%) – và nhóm quốc doanh vừa có CAR thấp vừa có NIM thấp (kết quả câu 2) → tạo ra tương quan âm.

In [9]:
q4_nhom = q4.groupby(["nam", "loai_hinh_so_huu"], observed=True)[Y].mean().unstack().round(1)
q4_nhom["Chênh lệch QD − TN"] = (q4_nhom["Quốc doanh"] - q4_nhom["Tư nhân"]).round(1)
q4_nhom

loai_hinh_so_huu,Quốc doanh,Tư nhân,Chênh lệch QD − TN
nam,,,
2021,12.6,6.9,5.7
2022,13.5,-5.5,19.0
2023,0.9,-2.3,3.2
2024,-2.4,1.2,-3.6


In [10]:
hien(ve_hop_nhom(q4, Y,
                 "Quý 4/2021–2024: nhóm quốc doanh (CAR, NIM thấp) có lợi suất quý 4 cao hơn"),
     "p5_q4_loi_suat_theo_nhom")
hien(ve_phan_tan(b4[b4["car_cong_bo"]], "car", ghi_chu="Chỉ 32 quý 4 có CAR thật; các chấm lợi suất cao bên trái chủ yếu là BID, CTG (và STB 2021)"),
     "p5_phan_tan_car_cong_bo")

→ Kết luận: tương quan âm của CAR (và NIM) ở quý 4 chủ yếu phản ánh việc **nhóm quốc doanh tăng giá mạnh hơn trong phần lớn các quý 4 của giai đoạn này**, không phải bằng chứng rằng CAR thấp làm giá tăng. Không nên chọn CAR là "chỉ số liên hệ rõ nhất".

---
## 6. Xuất bảng kết quả

In [11]:
xuat_ket_qua(tq_tong, "tuong_quan")
print(f"Đã lưu outputs/tuong_quan.csv: {len(tq_tong)} dòng "
      f"({(tq_tong['phan_tich'] == 'chinh').sum()} chính + {(tq_tong['phan_tich'] == 'kiem_tra_lai').sum()} kiểm tra lại)")

Đã lưu outputs/tuong_quan.csv: 31 dòng (6 chính + 25 kiểm tra lại)


---
## 7. Trả lời câu 3 (bản nháp – viết lại bằng lời của bạn)

**Không có chỉ số tài chính nào liên hệ rõ với biến động giá cổ phiếu trong cùng quý.** Trên 160 quan sát ngân hàng–quý, cả 6 chỉ số đều có tương quan rất yếu với lợi suất quý (|r|, |ρ| ≤ 0,09) và không chỉ số nào có ý nghĩa thống kê, kể cả trước khi hiệu chỉnh nhiều kiểm định. Nếu buộc phải xếp hạng, **nhóm chỉ số sinh lời (ROE, ROA, NIM) có liên hệ dương mạnh nhất** – ROE đứng đầu theo Spearman (ρ = 0,08), ROA theo Pearson (r = 0,09) – và nhóm này giữ vị trí đầu khi loại bỏ biến động chung của ngành (r = 0,11–0,15), nhưng mức liên hệ vẫn rất yếu. NPL và CAR gần như không liên hệ (hệ số ≈ 0).

**Vì sao yếu?** Khoảng 66% biến động lợi suất quý là biến động chung: trong cùng một quý, 8 cổ phiếu ngân hàng thường cùng tăng hoặc cùng giảm theo thị trường. Phần riêng của từng ngân hàng – phần mà chỉ số tài chính có thể giải thích – chỉ chiếm khoảng một phần ba. Phần 2 cũng cho thấy điều này: STB có ROE thấp nhưng giá tăng mạnh nhất, VCB có ROE cao nhưng giá tăng ít nhất.

**Riêng CAR:** trên 32 quý có số CAR thật, CAR có tương quan âm vừa phải (ρ = −0,52, có ý nghĩa). Tuy nhiên NIM và LDR trên cùng mẫu cũng âm gần bằng, CAR công bố sau khi quý 4 kết thúc, và tương quan này chủ yếu phản ánh việc cổ phiếu quốc doanh (CAR thấp) tăng mạnh hơn trong 3/4 quý 4 – nên không coi là bằng chứng CAR liên hệ với giá.

**Hạn chế và lưu ý.**
1. Tương quan không phải nhân quả; đây là quan hệ cùng kỳ, không phải khả năng dự báo (thuộc câu 4).
2. Chỉ 8 ngân hàng, 20 quý; các quan sát của cùng một ngân hàng không độc lập hoàn toàn.
3. Lợi suất tính theo quý lịch, trong khi BCTC quý công bố khoảng 1–1,5 tháng sau khi quý kết thúc – thị trường có thể phản ứng với số liệu ở quý sau.
4. **Gợi ý cho Quân (câu 4):** chỉ số tài chính đơn lẻ giải thích rất ít lợi suất; mô hình dự báo nên thử kết hợp nhiều biến và biến động thị trường chung, và nên kỳ vọng độ chính xác khiêm tốn.